# Modèle Underfitting (sous-apprentissage)

Objectif : illustrer le **sous-apprentissage**. On entraîne volontairement une forêt **trop simple** (arbres de profondeur 2), incapable de capter le signal.

## 1. Pré-traitement (repris de `preprocessing.ipynb`)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [2]:
data = pd.read_csv('data/kc_house_data.csv')
data = data.drop('id', axis=1)
data = data.drop('sqft_above', axis=1)  # colinéaire avec sqft_living

In [3]:
# Feature engineering : âge du bien + binarisation de la rénovation
data['date'] = pd.to_datetime(data['date'], format='%Y%m%dT%H%M%S')
data['yr_sold'] = data['date'].dt.year
data['house_age'] = data['yr_sold'] - data['yr_built']
data['was_renovated'] = (data['yr_renovated'] > 0).astype(int)
data = data.drop(['date', 'yr_built', 'yr_renovated'], axis=1)

In [4]:
# Split stratifié (cible condition très déséquilibrée)
X = data.drop('condition', axis=1)
y = data['condition']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((17290, 18), (4323, 18))

In [5]:
def evaluate(model):
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    return pd.Series({
        'acc_train': accuracy_score(y_train, pred_train),
        'acc_test': accuracy_score(y_test, pred_test),
        'f1_train': f1_score(y_train, pred_train, average='macro'),
        'f1_test': f1_score(y_test, pred_test, average='macro'),
    }).round(3)

## 2. Modèle volontairement trop simple

`max_depth=2` bride fortement chaque arbre : le modèle n'a pas assez de capacité pour apprendre.

In [6]:
rf_underfit = RandomForestClassifier(
    n_estimators=50, max_depth=2, random_state=42
)
rf_underfit.fit(X_train, y_train)
evaluate(rf_underfit)

acc_train    0.649
acc_test     0.649
f1_train     0.157
f1_test      0.157
dtype: float64

## 3. Diagnostic

**Signature du sous-apprentissage : les scores sont bas au train ET au test** (`f1` faible partout).
Le modèle est trop contraint pour apprendre, donc il se trompe même sur les données d'entraînement.
C'est un problème de **biais élevé** : la solution est un modèle plus complexe (arbres plus profonds, plus d'estimateurs), pas plus de données.